# Week 5, Day 6: Forge: Week 5 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w005/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", (np.ones((3, 4)) @ np.ones((4, 2))).shape)
M = np.array([[2, 4], [1, 2.0]])
print("2.", round(np.linalg.det(M), 6), "so no inverse: the second column is twice the first")
X = np.column_stack([np.ones(3), [0, 1, 2]])
print("3. intercept and slope:", np.linalg.lstsq(X, np.array([1, 2, 4.0]), rcond=None)[0].round(3))

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(2011)
n_train, n_test, p, V = 120, 40, 150, 30
mix = rng.normal(size=(p, p)) * 0.5 + np.eye(p)                  # makes the features correlated
F_train = rng.normal(size=(n_train, p)) @ mix / np.sqrt(p) * 3     # (120, 150) image features
F_test = rng.normal(size=(n_test, p)) @ mix / np.sqrt(p) * 3       # (40, 150)
W_true = rng.normal(size=(p, V)) * (rng.random((p, V)) < 0.2)      # each voxel uses about 30 features
scale = (F_train @ W_true).std(axis=0)
Y_train = F_train @ W_true + rng.normal(0, 1, (n_train, V)) * scale  # (120, 30) voxel responses
Y_test = F_test @ W_true + rng.normal(0, 1, (n_test, V)) * scale     # (40, 30)

def voxel_r(Y, P):
    """Correlation between measured and predicted responses, one value per voxel (column)."""
    Yc, Pc = Y - Y.mean(axis=0), P - P.mean(axis=0)
    return (Yc * Pc).sum(axis=0) / np.sqrt((Yc ** 2).sum(axis=0) * (Pc ** 2).sum(axis=0))

print(F_train.shape, Y_train.shape, "rank of F_train:", np.linalg.matrix_rank(F_train))

### Exercise: Part A: every voxel in one equation

Write `ridge(X, Y, lam)` that solves $(X^\top X + \lambda I)B = X^\top Y$ for a whole matrix of targets at once (one column per voxel). Then write `fit_predict(Ftr, Ytr, Fte, lam)`: centre features and responses with the training means, fit ridge, and return predictions for `Fte` with the response means added back.

In [ ]:
def ridge(X, Y, lam):
    ...

def fit_predict(Ftr, Ytr, Fte, lam):
    ...

print(fit_predict(F_train, Y_train, F_test, 100.0).shape)

In [ ]:
import numpy as np
def test_shape():
    "ridge returns one column of weights per voxel"
    assert ridge(np.ones((5, 3)), np.ones((5, 4)), 1.0).shape == (3, 4)

def test_columns():
    "fitting all voxels at once matches fitting each voxel on its own"
    X = np.random.default_rng(1).normal(size=(40, 6)); Y = np.random.default_rng(2).normal(size=(40, 3))
    B = ridge(X, Y, 2.0)
    for v in range(3):
        b = np.linalg.solve(X.T @ X + 2.0 * np.eye(6), X.T @ Y[:, v])
        assert np.allclose(B[:, v], b)

def test_fit_predict():
    "fit_predict centres with the training means and returns predictions for the new rows"
    P = fit_predict(F_train, Y_train, F_test, 100.0)
    assert P.shape == (40, 30)
    assert np.allclose(fit_predict(F_train, Y_train, F_train[:1] * 0 + F_train.mean(axis=0), 100.0), Y_train.mean(axis=0))

test_shape()
test_columns()
test_fit_predict()
print("All checks passed")

### Exercise: Part B: choose the penalty, then test once

Run 5-fold cross-validation on the training images for each penalty in `lams`, scoring each fold by the mean over voxels of `voxel_r`. Store the averages in `cv_score` and the winner in `best_lam`. Then refit on all 120 training images, predict the 40 test images, and store each voxel's held-out correlation in `r_test`. Compare with a near-zero penalty.

In [ ]:
lams = [1, 10, 100, 1000, 10000, 100000]
folds = np.array_split(np.random.default_rng(0).permutation(n_train), 5)
cv_score = []
for lam in lams:
    scores = []
    for f in folds:
        tr = np.setdiff1d(np.arange(n_train), f)
        ...
    cv_score.append(np.mean(scores))
best_lam = ...
P_test = ...
r_test = ...

r_small = voxel_r(Y_test, fit_predict(F_train, Y_train, F_test, 1e-6))
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].semilogx(lams, cv_score, "o-"); ax[0].axvline(best_lam, ls="--", color="0.5")
ax[0].set_xlabel("ridge penalty λ"); ax[0].set_ylabel("cross-validated mean r")
ax[1].hist(r_small, bins=12, alpha=0.6, label="λ near 0"); ax[1].hist(r_test, bins=12, alpha=0.6, label=f"λ = {best_lam}")
ax[1].set_xlabel("held-out r per voxel"); ax[1].legend(frameon=False); plt.tight_layout(); plt.show()
print(f"best lambda {best_lam}: mean held-out r = {r_test.mean():.2f} (near-zero penalty: {r_small.mean():.2f})")

In [ ]:
import numpy as np
def test_cv():
    "cv_score has one mean cross-validated correlation per penalty"
    assert len(cv_score) == len(lams)

def test_best():
    "best_lam maximises the cross-validated score"
    assert best_lam == lams[int(np.argmax(cv_score))]

def test_accuracy():
    "held-out prediction accuracy r_test has one value per voxel and averages above 0.35"
    assert r_test.shape == (30,)
    assert r_test.mean() > 0.35, f"mean r = {r_test.mean():.2f}"

test_cv()
test_best()
test_accuracy()
print("All checks passed")

### Exercise: Part C: which image was it?

Identification, as in Kay and colleagues (2008): for each test image `j`, compare the measured pattern across the 30 voxels, `Y_test[j]`, with the predicted pattern for every test image `k`, `P_test[k]`. Fill `C[j, k]` with their correlation, pick `identified[j]` as the best match, and compute `accuracy`, the fraction identified correctly. Chance is 1 in 40.

In [ ]:
C = ...
identified = ...
accuracy = ...

plt.figure(figsize=(3.8, 3.2)); plt.imshow(C, cmap="viridis"); plt.colorbar(label="r")
plt.xlabel("predicted image k"); plt.ylabel("measured image j"); plt.show()
print(f"identified {accuracy:.0%} of the test images correctly (chance {1 / n_test:.1%})")

In [ ]:
import numpy as np
def test_C():
    "C is the 40 by 40 matrix of correlations between measured and predicted patterns"
    assert C.shape == (40, 40)
    assert np.isclose(C[3, 7], np.corrcoef(Y_test[3], P_test[7])[0, 1])

def test_identified():
    "identified holds the best-matching predicted image for each measured pattern"
    assert np.array_equal(identified, C.argmax(axis=1))

def test_accuracy():
    "identification is far above chance (1 in 40)"
    assert accuracy > 0.4, f"accuracy {accuracy:.2f}"

test_C()
test_identified()
test_accuracy()
print("All checks passed")